Importing Libraries


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import optuna

from tensorflow import keras
from sklearn.metrics import (
    precision_recall_curve, auc,
    f1_score, precision_score, recall_score,
    confusion_matrix
)

d:\Fraud-Detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Data and models

In [3]:
# Loading MODELS
xgb_best    = joblib.load('../models/xgboost_best.pkl')
autoencoder = keras.models.load_model('../models/autoencoder.keras')
scaler      = joblib.load('../models/Standard_scaler.pkl')

# Loading thresholds
xgb_thresholds = joblib.load('../models/xgb_threshold.pkl')
ae_config      = joblib.load('../models/ae_threshold.pkl')

best_threshold_xgb = xgb_thresholds['xgb_threshold']
best_threshold_ae  = ae_config['ae_threshold']
clip_val           = ae_config['ae_clip_val']

# Loading Data
X_val  = pd.read_parquet('../data/processed/X_val.parquet')
X_test = pd.read_parquet('../data/processed/X_test.parquet')
y_val  = pd.read_parquet('../data/processed/y_val.parquet')['isFraud']
y_test = pd.read_parquet('../data/processed/y_test.parquet')['isFraud']

First computing individial scores of both the models

In [5]:
# XGBoost fraud probabilities
xgb_proba_val  = xgb_best.predict_proba(X_val)[:, 1]
xgb_proba_test = xgb_best.predict_proba(X_test)[:, 1]

# Autoencoder reconstruction errors
X_val_scaled  = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

X_val_clipped  = np.clip(X_val_scaled.values,  -clip_val, clip_val)
X_test_clipped = np.clip(X_test_scaled.values, -clip_val, clip_val)

ae_errors_val  = np.mean((X_val_clipped  - autoencoder.predict(X_val_clipped,  verbose=0)) ** 2 , axis=1)
ae_errors_test = np.mean((X_test_clipped - autoencoder.predict(X_test_clipped, verbose=0)) ** 2,axis=1)

# Using 99th percentile method to normalize it , for a more robust normalization

ae_p1  = np.percentile(ae_errors_val, 1)
ae_p99 = np.percentile(ae_errors_val, 99)

ae_errors_val_clipped  = np.clip(ae_errors_val,  ae_p1, ae_p99)
ae_errors_test_clipped = np.clip(ae_errors_test, ae_p1, ae_p99)

ae_norm_val  = (ae_errors_val_clipped  - ae_p1) / (ae_p99 - ae_p1)
ae_norm_test = (ae_errors_test_clipped - ae_p1) / (ae_p99 - ae_p1)


print(f"XGBoost proba  — mean: {xgb_proba_val.mean():.4f}, std: {xgb_proba_val.std():.4f}")
print(f"AE norm errors — mean: {ae_norm_val.mean():.4f},  std: {ae_norm_val.std():.4f}")
print(f"\nFraud XGBoost proba mean:  {xgb_proba_val[y_val==1].mean():.4f}")
print(f"Legit XGBoost proba mean:  {xgb_proba_val[y_val==0].mean():.4f}")
print(f"\nFraud AE norm error mean:  {ae_norm_val[y_val==1].mean():.4f}")
print(f"Legit AE norm error mean:  {ae_norm_val[y_val==0].mean():.4f}")

XGBoost proba  — mean: 0.1141, std: 0.1957
AE norm errors — mean: 0.0437,  std: 0.1280

Fraud XGBoost proba mean:  0.6219
Legit XGBoost proba mean:  0.0928

Fraud AE norm error mean:  0.1378
Legit AE norm error mean:  0.0398


In [6]:
ae_config['ae_p1']  = float(ae_p1)
ae_config['ae_p99'] = float(ae_p99)
joblib.dump(ae_config, '../models/ae_threshold.pkl')

['../models/ae_threshold.pkl']

Optuna tuning to find best ensemble methods

In [8]:
def ensemble_objective(trial):
    w1 = trial.suggest_float('w1' , 0.0 , 1.0)
    w2 = 1.0 - w1

    ensemble_score = w1 * xgb_proba_val + w2 * ae_norm_val

    pr_prec, pr_rec, _ = precision_recall_curve(y_val, ensemble_score)
    return auc(pr_rec, pr_prec)

study_ensemble = optuna.create_study(direction='maximize' , sampler=optuna.samplers.TPESampler(seed=42))

# Seed with XGBoost-only as first trial
study_ensemble.enqueue_trial({'w1': 1.0})
# Seed with equal weights as second trial
study_ensemble.enqueue_trial({'w1': 0.5})

study_ensemble.optimize(ensemble_objective, n_trials=100, show_progress_bar=True)

best_w1 = study_ensemble.best_params['w1']
best_w2 = 1.0 - best_w1

[I 2026-06-11 11:37:55,284] A new study created in memory with name: no-name-8fba7c53-7a9a-4f7c-8e74-fa54e2e5a643
Best trial: 3. Best value: 0.591181:   8%|▊         | 8/100 [00:00<00:01, 46.29it/s]

[I 2026-06-11 11:37:55,312] Trial 0 finished with value: 0.5841082948442594 and parameters: {'w1': 1.0}. Best is trial 0 with value: 0.5841082948442594.
[I 2026-06-11 11:37:55,335] Trial 1 finished with value: 0.4713245588929113 and parameters: {'w1': 0.5}. Best is trial 0 with value: 0.5841082948442594.
[I 2026-06-11 11:37:55,356] Trial 2 finished with value: 0.3711338251408773 and parameters: {'w1': 0.3745401188473625}. Best is trial 0 with value: 0.5841082948442594.
[I 2026-06-11 11:37:55,376] Trial 3 finished with value: 0.5911809907155972 and parameters: {'w1': 0.9507143064099162}. Best is trial 3 with value: 0.5911809907155972.
[I 2026-06-11 11:37:55,395] Trial 4 finished with value: 0.5815741191072054 and parameters: {'w1': 0.7319939418114051}. Best is trial 3 with value: 0.5911809907155972.
[I 2026-06-11 11:37:55,414] Trial 5 finished with value: 0.5588244955369938 and parameters: {'w1': 0.5986584841970366}. Best is trial 3 with value: 0.5911809907155972.
[I 2026-06-11 11:37:55

Best trial: 13. Best value: 0.591778:  17%|█▋        | 17/100 [00:00<00:01, 48.31it/s]

[I 2026-06-11 11:37:55,490] Trial 9 finished with value: 0.5916915399408302 and parameters: {'w1': 0.8661761457749352}. Best is trial 9 with value: 0.5916915399408302.
[I 2026-06-11 11:37:55,514] Trial 10 finished with value: 0.587310408730117 and parameters: {'w1': 0.7772371258314931}. Best is trial 9 with value: 0.5916915399408302.
[I 2026-06-11 11:37:55,536] Trial 11 finished with value: 0.5908914548510175 and parameters: {'w1': 0.9588266146737304}. Best is trial 9 with value: 0.5916915399408302.
[I 2026-06-11 11:37:55,557] Trial 12 finished with value: 0.5896953328008453 and parameters: {'w1': 0.8162367438939515}. Best is trial 9 with value: 0.5916915399408302.
[I 2026-06-11 11:37:55,576] Trial 13 finished with value: 0.5917776297350589 and parameters: {'w1': 0.8704936397746927}. Best is trial 13 with value: 0.5917776297350589.
[I 2026-06-11 11:37:55,595] Trial 14 finished with value: 0.570451648981566 and parameters: {'w1': 0.6465959674006365}. Best is trial 13 with value: 0.59177

[I 2026-06-11 11:37:55,680] Trial 18 finished with value: 0.4582511744345191 and parameters: {'w1': 0.4946443014220543}. Best is trial 13 with value: 0.5917776297350589.
[I 2026-06-11 11:37:55,700] Trial 19 finished with value: 0.5822801338776395 and parameters: {'w1': 0.7384055447250206}. Best is trial 13 with value: 0.5917776297350589.
[I 2026-06-11 11:37:55,720] Trial 20 finished with value: 0.5918812656531243 and parameters: {'w1': 0.8869137510949834}. Best is trial 20 with value: 0.5918812656531243.
[I 2026-06-11 11:37:55,740] Trial 21 finished with value: 0.5917577041208424 and parameters: {'w1': 0.869788751966085}. Best is trial 20 with value: 0.5918812656531243.
[I 2026-06-11 11:37:55,761] Trial 22 finished with value: 0.591859496006673 and parameters: {'w1': 0.9041509459720934}. Best is trial 20 with value: 0.5918812656531243.
[I 2026-06-11 11:37:55,781] Trial 23 finished with value: 0.5917137516390053 and parameters: {'w1': 0.9241806734875834}. Best is trial 20 with value: 0.

...

[I 2026-06-11 11:37:56,819] Trial 73 finished with value: 0.5893475894952227 and parameters: {'w1': 0.9882200265451712}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:56,839] Trial 74 finished with value: 0.5895652509680968 and parameters: {'w1': 0.8124044023053298}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:56,858] Trial 75 finished with value: 0.5919211365664899 and parameters: {'w1': 0.9112476087288826}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:56,878] Trial 76 finished with value: 0.5918841540036228 and parameters: {'w1': 0.9128177171493425}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:56,899] Trial 77 finished with value: 0.5854159681570541 and parameters: {'w1': 0.7622433052028046}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:56,919] Trial 78 finished with value: 0.5915295802785167 and parameters: {'w1': 0.8602707031878772}. Best is trial 51 with value: 

Best trial: 51. Best value: 0.591938:  92%|█████████▏| 92/100 [00:01<00:00, 49.21it/s]

[I 2026-06-11 11:37:57,001] Trial 82 finished with value: 0.3318918186431909 and parameters: {'w1': 0.29363528329077404}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:57,023] Trial 83 finished with value: 0.5919362542626024 and parameters: {'w1': 0.8939696039484707}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:57,043] Trial 84 finished with value: 0.5916429645907338 and parameters: {'w1': 0.9266906869959414}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:57,063] Trial 85 finished with value: 0.590511010349282 and parameters: {'w1': 0.8338891962140212}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:57,084] Trial 86 finished with value: 0.5895877538267407 and parameters: {'w1': 0.9763013362636419}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:57,105] Trial 87 finished with value: 0.591739213817171 and parameters: {'w1': 0.8686317201863791}. Best is trial 51 with value: 0

Best trial: 51. Best value: 0.591938: 100%|██████████| 100/100 [00:02<00:00, 48.42it/s]

[I 2026-06-11 11:37:57,205] Trial 92 finished with value: 0.5914311454148705 and parameters: {'w1': 0.9362667617559715}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:57,227] Trial 93 finished with value: 0.5906391522656328 and parameters: {'w1': 0.9629330342406126}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:57,249] Trial 94 finished with value: 0.591673910521158 and parameters: {'w1': 0.8654699241508108}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:57,270] Trial 95 finished with value: 0.5919351798890171 and parameters: {'w1': 0.8920438674883102}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:57,292] Trial 96 finished with value: 0.5869214874980772 and parameters: {'w1': 0.9982096062436346}. Best is trial 51 with value: 0.5919378504412888.
[I 2026-06-11 11:37:57,312] Trial 97 finished with value: 0.5919119461514016 and parameters: {'w1': 0.9115712411961758}. Best is trial 51 with value: 0

In [9]:
pr_prec_xgb, pr_rec_xgb, _ = precision_recall_curve(y_val, xgb_proba_val)
pr_prec_ae,  pr_rec_ae,  _ = precision_recall_curve(y_val, ae_norm_val)
pr_auc_xgb = auc(pr_rec_xgb, pr_prec_xgb)
pr_auc_ae  = auc(pr_rec_ae,  pr_prec_ae)

print(f"\nBest weights: XGBoost={best_w1:.4f}, Autoencoder={best_w2:.4f}")
print(f"\nXGBoost alone PR-AUC:  {pr_auc_xgb:.4f}")
print(f"Autoencoder alone PR-AUC: {pr_auc_ae:.4f}")
print(f"Ensemble PR-AUC:       {study_ensemble.best_value:.4f}")


Best weights: XGBoost=0.8933, Autoencoder=0.1067

XGBoost alone PR-AUC:  0.5841
Autoencoder alone PR-AUC: 0.1200
Ensemble PR-AUC:       0.5919


Validating the ensemble model

In [11]:
ensemble_score_val = best_w1 * xgb_proba_val + best_w2 * ae_norm_val
pr_prec_ens, pr_rec_ens, thresholds_ens = precision_recall_curve(y_val , ensemble_score_val)

f1_scores = 2 * (pr_prec_ens * pr_rec_ens) / (pr_prec_ens + pr_rec_ens + 1e-8)
best_idx  = f1_scores.argmax()
best_threshold_ensemble = thresholds_ens[best_idx]
pr_auc_ensemble = auc(pr_rec_ens, pr_prec_ens)

# Evaluate
ens_preds = (ensemble_score_val >= best_threshold_ensemble).astype(int)
precision = precision_score(y_val, ens_preds, zero_division=0)
recall    = recall_score(y_val, ens_preds, zero_division=0)
f1        = f1_score(y_val, ens_preds, zero_division=0)
cm        = confusion_matrix(y_val, ens_preds)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn)

In [12]:
print(f"PR-AUC:    {pr_auc_ensemble:.4f}")
print(f"Threshold: {best_threshold_ensemble:.4f}")
print(f"F1:        {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"FPR:       {fpr:.4f} ({fpr*100:.2f}% legitimate transactions blocked)")
print(f"\nConfusion Matrix:")
print(f"  TP: {tp:,}  FP: {fp:,}")
print(f"  FN: {fn:,}  TN: {tn:,}")

PR-AUC:    0.5919
Threshold: 0.7533
F1:        0.5823
Precision: 0.7603
Recall:    0.4719
FPR:       0.0063 (0.63% legitimate transactions blocked)

Confusion Matrix:
  TP: 1,687  FP: 532
  FN: 1,888  TN: 84,474


FINAL EVALUATION ON TEST SET 

In [13]:
ensemble_score_test = best_w1 * xgb_proba_test + best_w2 * ae_norm_test
ens_test_preds = (ensemble_score_test >= best_threshold_ensemble).astype(int)

precision_t = precision_score(y_test, ens_test_preds, zero_division=0)
recall_t    = recall_score(y_test, ens_test_preds, zero_division=0)
f1_t        = f1_score(y_test, ens_test_preds, zero_division=0)
pr_prec_t, pr_rec_t, _ = precision_recall_curve(y_test, ensemble_score_test)
pr_auc_t    = auc(pr_rec_t, pr_prec_t)
cm_t        = confusion_matrix(y_test, ens_test_preds)
tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
fpr_t = fp_t / (fp_t + tn_t)

total_fraud = y_test.sum()
total_legit = (y_test==0).sum()

In [15]:
print(f"PR-AUC:    {pr_auc_t:.4f}")
print(f"F1:        {f1_t:.4f}")
print(f"Precision: {precision_t:.4f}")
print(f"Recall:    {recall_t:.4f}")
print(f"FPR:       {fpr_t:.4f} ({fpr_t*100:.2f}% legitimate transactions blocked)")
print(f"\nConfusion Matrix:")
print(f"  TP: {tp_t:,}  FP: {fp_t:,}")
print(f"  FN: {fn_t:,}  TN: {tn_t:,}")
print(f"\nBusiness Impact:")

print(f"  Total fraud transactions:   {total_fraud:,}")
print(f"  Fraud caught (TP):          {tp_t:,} ({tp_t/total_fraud*100:.1f}%)")
print(f"  Fraud missed (FN):          {fn_t:,} ({fn_t/total_fraud*100:.1f}%)")
print(f"  Total legit transactions:   {total_legit:,}")
print(f"  Wrongly blocked (FP):       {fp_t:,} ({fpr_t*100:.2f}%)")

PR-AUC:    0.4200
F1:        0.4618
Precision: 0.6756
Recall:    0.3508
FPR:       0.0060 (0.60% legitimate transactions blocked)

Confusion Matrix:
  TP: 1,789  FP: 859
  FN: 3,311  TN: 141,676

Business Impact:
  Total fraud transactions:   5,100
  Fraud caught (TP):          1,789 (35.1%)
  Fraud missed (FN):          3,311 (64.9%)
  Total legit transactions:   142,535
  Wrongly blocked (FP):       859 (0.60%)


SAVING THE FINAL MODEL

In [16]:
ensemble_config = {
    'w1_xgb':             float(best_w1),
    'w2_ae':              float(best_w2),
    'ensemble_threshold': float(best_threshold_ensemble),
    'ae_norm_min':        float(ae_p1),
    'ae_norm_max':        float(ae_p99),
    'ae_clip_val':        float(clip_val)
}
ensemble_config

{'w1_xgb': 0.8933205867415125,
 'w2_ae': 0.10667941325848751,
 'ensemble_threshold': 0.7533466936605465,
 'ae_norm_min': 0.0004812768170920437,
 'ae_norm_max': 0.3752024123203878,
 'ae_clip_val': 10.0}

In [17]:
joblib.dump(ensemble_config, '../models/ensemble_config.pkl')

['../models/ensemble_config.pkl']